# DDI Atomic Triplet Extraction — v4

## Root-cause diagnosis of v3 failures

v3 produced only **58 triplets from 50 responses** (26 responses = zero triplets, ~1.2 triplets/response).  
Three failure modes were identified:

| Failure | Example | Root Cause |
|---|---|---|
| **Pronoun subject** | `"it doesn't interact with lamivudine"` | `it` is the grammatical nsubj — no drug token on subj side, verb-centric extraction fires nothing |
| **OOV drugs** | `"Fibrates"`, `"HMG-CoA reductase inhibitors"`, `"ethanol"` | Not in FDA NDC PhraseMatcher → invisible, sentence drops below 2-drug threshold |
| **Self-pairs** | e1 == e2 (degenerate rows) | Nothing to extract, expected |

## v4 fixes: three-tier extraction

Since we are in a **supervised NLI setting** — every holistic response has a known reference pair  
(`e1_text`, `e2_text`) from the DDI-2013 corpus — we exploit this to fix both failure modes:

**Tier 1 — Explicit (both drugs appear as tokens):** Same verb-centric extraction as v3.  
**Tier 2 — OOV fix:** `e1_text` and `e2_text` are injected as per-row matcher patterns,  
guaranteeing recognition even for drug class names and trade names not in the FDA NDC index.  
**Tier 3 — Pronoun coreference:** When a sentence has exactly 1 matched drug token and  
its governing verb's grammatical subject is a pronoun (`it`, `they`, `its`, `their`, `this`),  
the pronoun is resolved to the known partner drug from the reference pair. The verb and  
negation are extracted normally.

The `source_tier` column records which tier produced each triplet for interpretability.

In [1]:
import pandas as pd
import spacy
from spacy.matcher import PhraseMatcher
from collections import defaultdict
from tqdm.notebook import tqdm
import networkx as nx
import scispacy

In [2]:
df_holistic = pd.read_csv("synthetic_rag_dataset_groq.csv")
df_raw_fda  = pd.read_csv("drug_products.csv", encoding="ISO-8859-1")

nlp = spacy.load("en_core_web_sm")

# ── Build global FDA drug name set ──────────────────────────────────────────
df_prescription = df_raw_fda[df_raw_fda["PRODUCTTYPENAME"] == "HUMAN PRESCRIPTION DRUG"]

fda_drug_names = set()
for col in ["PROPRIETARYNAME", "NONPROPRIETARYNAME", "SUBSTANCENAME"]:
    if col in df_prescription.columns:
        for val in df_prescription[col].dropna().unique():
            val_clean = str(val).lower().strip()
            for part in val_clean.split(";"):
                fda_drug_names.add(part.strip())

global_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
global_patterns = [nlp.make_doc(t) for t in fda_drug_names if len(t.strip()) > 2]
global_matcher.add("FDA_NDC_INDEX", global_patterns)

print(f"Global drug vocabulary: {len(fda_drug_names):,} entries")

Global drug vocabulary: 8,165 entries


In [3]:
# ── Discourse/copular verbs — carry NO DDI information ───────────────────────
DISCOURSE_VERBS = {
    # existing
    "be", "say", "find", "read", "look", "talk", "go", "come",
    "know", "seem", "use", "make", "get", "have", "think", "check",
    "tell", "show", "mean", "note", "mention", "see", "hear",
    "explain", "describe", "share", "post", "write", "ask",
    # ADD: research/reporting verbs that appear in RAG framing
    "suggest", "indicate", "report", "demonstrate", "conduct", "study",
    "investigate", "examine", "evaluate", "assess", "publish", "document",
    "observe", "identify", "confirm", "discuss", "state", "highlight",
    "reveal", "consider", "review", "summarize", "outline",
}

# Pronouns that signal coreference to the contextual drug subject
SUBJECT_PRONOUNS = {"it", "its", "they", "their", "them", "this", "that"}

# dobj nouns with no pharmacological value
NOISE_DOBJS = {
    "stuff", "info", "information", "thing", "something", "study",
    "interaction", "data", "result", "report", "paper", "article",
    "drug", "medication", "med", "it", "they", "this", "that",
}


def build_row_matcher(e1: str, e2: str):
    """
    Build a per-row PhraseMatcher that combines the global FDA NDC index
    with the known reference pair (e1, e2).

    This ensures drug class names ('Fibrates', 'HMG-CoA reductase inhibitors'),
    non-prescription substances ('ethanol'), and trade names ('ZETIA') are
    always recognised even when absent from the FDA NDC index.
    """
    row_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
    # Re-add all global patterns
    row_matcher.add("FDA_NDC_INDEX", global_patterns)

    # Inject the known pair as guaranteed patterns
    extra = set()
    for name in [e1, e2]:
        if name and len(name.strip()) > 2:
            extra.add(name.lower().strip())
    if extra:
        extra_patterns = [nlp.make_doc(n) for n in extra]
        row_matcher.add("KNOWN_PAIR", extra_patterns)

    return row_matcher, {n.lower().strip() for n in [e1, e2] if n and len(n.strip()) > 2}


def get_sdp(doc, tok_a, tok_b):
    """
    Shortest Dependency Path between two tokens in a parsed doc.
    Returns the list of Token objects on the path (including endpoints),
    or None if no path exists.
    """
    edges = []
    for token in doc:
        for child in token.children:
            edges.append((token.i, child.i))
            edges.append((child.i, token.i))  # undirected
    G = nx.Graph(edges)
    try:
        path_indices = nx.shortest_path(G, tok_a.i, tok_b.i)
        return [doc[i] for i in path_indices]
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return None


def sdp_to_relation(path_tokens):
    """
    Convert the middle tokens of an SDP (endpoints excluded) into a
    clean relation string. Lemmatises verbs, prepends 'not' for negation,
    keeps nouns and adpositions, drops determiners and punctuation.
    Returns None if nothing pharmacologically useful survives.
    """
    if len(path_tokens) < 3:          # endpoints only → no middle tokens
        return None
    middle = path_tokens[1:-1]        # strip the two drug tokens

    parts = []
    for tok in middle:
        # Capture negation that hangs off this token
        neg = any(c.dep_ == "neg" for c in tok.children)
        if tok.pos_ == "VERB":
            parts.append(("not " if neg else "") + tok.lemma_.lower())
        elif tok.pos_ in ("NOUN", "ADP", "PART"):
            parts.append(tok.text.lower())
        elif tok.dep_ == "neg":       # standalone 'not', 'never' on path
            pass                      # already handled above; skip to avoid duplication
        # Drop: DET, PUNCT, SPACE, PRON, etc.

    rel = " ".join(parts).strip()
    return rel if rel else None


def determine_direction(drug_a, drug_b):
    """
    Heuristic for subject/object assignment.
    Prefers syntactic dependency labels; falls back to linear order.
    """
    SUBJ_DEPS = {"nsubj", "nsubjpass", "csubj", "agent"}
    if drug_a.dep_ in SUBJ_DEPS:
        return drug_a, drug_b
    if drug_b.dep_ in SUBJ_DEPS:
        return drug_b, drug_a
    # Linear order fallback
    return (drug_a, drug_b) if drug_a.i < drug_b.i else (drug_b, drug_a)


def build_hypothesis(subj_text, relation, obj_text):
    """
    Reconstruct a natural language hypothesis sentence for PubMedBERT NLI input.
    e.g. ("warfarin", "increases plasma level of", "aspirin")
         → "Warfarin increases plasma level of aspirin."
    """
    return f"{subj_text.capitalize()} {relation} {obj_text.lower()}."


print("Helpers defined.")

Helpers defined.


In [6]:
SUBJ_DEPS = {"nsubj", "nsubjpass", "csubj", "agent"}
OBJ_DEPS  = {"dobj", "pobj", "attr", "oprd"}


def extract_explicit(sent_doc, drug_tokens, drug_idx, e1, e2):
    """
    TIER 1: Both drugs appear as matched tokens in this sentence.
    Uses SDP instead of governing-verb extraction.
    Returns list of (subj, relation, obj, hypothesis, tier) tuples.
    """
    triplets = []
    seen_pairs = set()

    # Get all pairs of distinct drug tokens
    for i, ta in enumerate(drug_tokens):
        for tb in drug_tokens[i+1:]:
            if ta.text.lower() == tb.text.lower():
                continue

            pair_key = tuple(sorted([ta.i, tb.i]))
            if pair_key in seen_pairs:
                continue
            seen_pairs.add(pair_key)

            path = get_sdp(sent_doc, ta, tb)
            if path is None:
                continue

            relation = sdp_to_relation(path)
            if not relation:
                continue

            subj_tok, obj_tok = determine_direction(ta, tb)
            subj_name = subj_tok.text.lower()
            obj_name  = obj_tok.text.lower()

            hypothesis = build_hypothesis(subj_name, relation, obj_name)
            triplets.append((subj_name, relation, obj_name, hypothesis, "tier1_sdp"))

    return triplets


def extract_pronoun_coreference(sent_doc, drug_tokens, known_pair):
    """
    TIER 3: Exactly 1 drug found; pronoun subject resolved to partner drug.
    Uses SDP from the found drug to the governing verb token as a fallback
    when no second entity is present.
    """
    triplets = []
    if len(drug_tokens) != 1:
        return triplets

    found_drug = drug_tokens[0].text.lower()
    partner_drugs = [d for d in sorted(known_pair) if d != found_drug]
    if not partner_drugs:
        return triplets
    partner = partner_drugs[0]

    drug_tok = drug_tokens[0]
    gov = None
    current = drug_tok.head
    visited = set()
    while current.i not in visited:
        visited.add(current.i)
        if current.pos_ in ("VERB", "AUX"):
            gov = current
            break
        if current.head.i == current.i:
            break
        current = current.head

    if gov is None or gov.lemma_.lower() in DISCOURSE_VERBS:
        return triplets

    # Check for pronoun subject
    pronoun_subj = None
    for child in gov.children:
        if child.dep_ in ("nsubj", "nsubjpass") and child.text.lower() in SUBJECT_PRONOUNS:
            pronoun_subj = child
            break

    if pronoun_subj is None:
        return triplets

    # Build relation from verb (SDP not applicable; only one real drug)
    neg = any(c.dep_ == "neg" for c in gov.children)
    rel_parts = [("not " if neg else "") + gov.lemma_.lower()]
    for child in gov.children:
        if child.dep_ == "dobj" and child.pos_ == "NOUN":
            rel_parts.append(child.lemma_.lower())
        if child.dep_ == "prep":
            rel_parts.append(child.text.lower())

    relation = " ".join(rel_parts)
    # Pronoun → partner is the subject, found drug is the object
    hypothesis = build_hypothesis(partner, relation, found_drug)
    triplets.append((partner, relation, found_drug, hypothesis, "tier3_pronoun"))

    return triplets


print("Extraction tiers defined.")

Extraction tiers defined.


In [8]:
# ── Main loop ────────────────────────────────────────────────────────────────
atomic_rows = []

# Added tqdm so you can see progress
for _, row in tqdm(df_holistic.iterrows(), total=len(df_holistic)):
    
    # 🔴 CHANGED: Mapping to the new synthetic output column
    text    = row["synthetic_rag_output"]
    premise = row["premise"]
    e1      = str(row["e1_text"]).strip() if pd.notna(row["e1_text"]) else ""
    e2      = str(row["e2_text"]).strip() if pd.notna(row["e2_text"]) else ""

    if pd.isna(text) or not text.strip():
        continue

    # Skip degenerate self-pairs
    if e1.lower() == e2.lower():
        continue

    # TIER 2 FIX: build a per-row matcher that guarantees e1 and e2 are recognised
    row_matcher, known_pair = build_row_matcher(e1, e2)

    doc = nlp(text)
    
    # Track if ANY triplets were found for this generation
    found_triplets = False 

    for sentence in doc.sents:
        sent_doc = nlp(sentence.text)

        # Match drugs in this sentence (using the row-level matcher)
        matches = row_matcher(sent_doc)
        seen_starts = {}
        for match_id, start, end in matches:
            if start not in seen_starts:
                seen_starts[start] = sent_doc[start]

        # Deduplicate by text
        seen_names = {}
        for tok in seen_starts.values():
            name = tok.text.lower()
            if name not in seen_names:
                seen_names[name] = tok
        drug_tokens = list(seen_names.values())
        drug_idx    = {t.i for t in drug_tokens}

        triplets = []

        if len(drug_tokens) >= 2:
            # TIER 1: explicit — both drugs present
            triplets = extract_explicit(sent_doc, drug_tokens, drug_idx, e1, e2)

        elif len(drug_tokens) == 1:
            # TIER 3: pronoun coreference — one drug found, look for pronoun subject
            triplets = extract_pronoun_coreference(sent_doc, drug_tokens, known_pair)

        for subj, rel, obj, hypothesis, tier in triplets:
            if not rel.strip():
                continue
            found_triplets = True
            atomic_rows.append({
                "original_id":        row["original_id"],
                "Entity1":            e1,
                "Entity2":            e2,
                "scenario":           row["scenario"],
                "premise":            premise,
                "holistic_generation": text,
                "sub_extract":        subj,
                "obj_extract":        obj,
                "rel_extract":        rel,
                "hypothesis_text":    hypothesis,   # ← NEW: NL sentence for PubMedBERT
                "source_tier":        tier,
            })

        # The failsafe for fake_drug / no-match remains the same:
        if not found_triplets:
            atomic_rows.append({
                "original_id":        row["original_id"],
                "Entity1":            e1,
                "Entity2":            e2,
                "scenario":           row["scenario"],
                "premise":            premise,
                "holistic_generation": text,
                "sub_extract":        None,
                "obj_extract":        None,
                "rel_extract":        None,
                "hypothesis_text":    None,
                "source_tier":        "filtered_out",
            })

df_atomic = pd.DataFrame(atomic_rows)
print(f"Total rows logged (including filtered) : {len(df_atomic):,}")
print(f"Holistic responses       : {len(df_holistic)}")
print(f"Avg triplets per response: {len(df_atomic.dropna(subset=['rel_extract']))/len(df_holistic):.1f}")
print()
print("Tier breakdown:")
print(df_atomic["source_tier"].value_counts().to_string())

# Verification print to see if the Fake Drug scenario was successfully filtered
print("\nSuccess Check: Did the failsafe catch the fake drug scenarios?")
print(df_atomic.groupby("scenario")["rel_extract"].count().to_string())

  0%|          | 0/11 [00:00<?, ?it/s]

Total rows logged (including filtered) : 31
Holistic responses       : 11
Avg triplets per response: 2.0

Tier breakdown:
source_tier
tier1_sdp       22
filtered_out     9

Success Check: Did the failsafe catch the fake drug scenarios?
scenario
contradiction     8
entailment       10
fake_drug         4
neutral           0


In [9]:
print("Relation distribution:")
print(df_atomic["rel_extract"].value_counts().head(30).to_string())
print()
print("Coverage — responses with ≥1 triplet:")
covered = df_atomic["holistic_generation"].nunique()
print(f"  {covered} / {len(df_holistic)} responses ({100*covered/len(df_holistic):.0f}%)")
print()
print("Sample triplets:")
df_atomic[["Entity1","Entity2","sub_extract","rel_extract","obj_extract","source_tier"]].head(20)

Relation distribution:
rel_extract
of addition not alter properties of                                  3
lamivudine of addition not alter properties of                       3
decrease levels of                                                   2
of properties not alter by addition of                               2
co administer with                                                   2
add to combination of                                                2
of presence increase concentrations of                               1
of pharmacokinetics remain with without combination of               1
of pharmacokinetics remain with without combination of lamivudine    1
of coadministration                                                  1
of pharmacokinetics remain with without                              1
of pharmacokinetics remain with without lamivudine                   1
of combination                                                       1
increase metabolism of                    

,Entity1,Entity2,sub_extract,rel_extract,obj_extract,source_tier
0,abacavir,lamivudine,lamivudine,of addition not alter properties of,abacavir,tier1_sdp
1,abacavir,lamivudine,zidovudine,lamivudine of addition not alter properties of,abacavir,tier1_sdp
2,abacavir,lamivudine,abacavir,decrease levels of,lamivudine,tier1_sdp
3,abacavir,lamivudine,lamivudine,of presence increase concentrations of,abacavir,tier1_sdp
4,abacavir,lamivudine,NaN,NaN,NaN,filtered_out
5,abacavir,lamivudine,NaN,NaN,NaN,filtered_out
6,abacavir,lamivudine,NaN,NaN,NaN,filtered_out
7,abacavir,lamivudine,abacavir,of properties not alter by addition of,zidovudine,tier1_sdp
8,abacavir,lamivudine,abacavir,co administer with,zidovudine,tier1_sdp
9,abacavir,zidovudine,lamivudine,of addition not alter properties of,abacavir,tier1_sdp


In [10]:
df_atomic.to_csv("extracted_atomic_triplets_v4.csv", index=False)
print("Saved to extracted_atomic_triplets_v4.csv")

Saved to extracted_atomic_triplets_v4.csv
